# 08.08 — Framework Validation

Run common split, holdout, preprocessing-policy, and target-availability checks.

In [1]:
# Import libraries
from pathlib import Path
import sys

In [2]:
# Define the root directory of the project
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / "configs" / "modeling_foundation.yaml"
CONFIG_PATH

WindowsPath('e:/jcuenca/OneDrive - GUSCanada/5toTerm/01_Capstone/Codigo/ontario-electricity-peak-risk/configs/modeling_foundation.yaml')

In [3]:
# Import module to manage modeling configuration, feature datasets, and directories
from src.ontario_peak_risk.modeling.common import (
    load_modeling_config,
    load_feature_dataset,
    ensure_modeling_directories,
)

In [4]:
# Load the modeling configuration, feature dataset, and ensure necessary directories exist
CONFIG, _ = load_modeling_config(CONFIG_PATH)
REPORTS_DIR, DOCS_DIR, OUTPUTS_DIR = ensure_modeling_directories(CONFIG, PROJECT_ROOT)
feature_dataset = load_feature_dataset(CONFIG, PROJECT_ROOT)
feature_dataset.shape

(262944, 100)

In [5]:
# Import modules for building validation folds and holdout fold, as well as validating temporal folds, holdout policy, and preprocessing policy
from src.ontario_peak_risk.modeling.splits import (
    build_validation_folds,
    build_final_holdout_fold,
)
from src.ontario_peak_risk.modeling.validation import (
    validate_temporal_folds,
    validate_holdout_policy,
    validate_preprocessing_policy,
)

In [ ]:
# Build validation folds and final holdout fold
validation_folds = build_validation_folds(CONFIG)
final_holdout = build_final_holdout_fold(CONFIG)
all_folds = [*validation_folds, final_holdout]

# Validate temporal folds using the feature dataset and the defined time and group columns from the configuration
temporal_validation = validate_temporal_folds(
    feature_dataset,
    all_folds,
    time_column=CONFIG["modeling"]["time"]["time_column"],
    group_column=CONFIG["modeling"]["time"]["group_column"],
)

# Display the results of the temporal validation
temporal_validation


,fold,role,check,status
0,fold_2023,validation,non_empty_train,PASS
1,fold_2023,validation,non_empty_evaluation,PASS
2,fold_2023,validation,train_precedes_evaluation,PASS
3,fold_2023,validation,same_fsa_coverage,PASS
4,fold_2023,validation,no_temporal_overlap,PASS
5,fold_2023,validation,train_origin_is_horizon_safe,PASS
6,fold_2023,validation,evaluation_origin_is_horizon_safe,PASS
7,fold_2024,validation,non_empty_train,PASS
8,fold_2024,validation,non_empty_evaluation,PASS
9,fold_2024,validation,train_precedes_evaluation,PASS


In [7]:
# Validate the holdout policy using the configuration
holdout_validation = validate_holdout_policy(CONFIG)
holdout_validation


,check,status,latest_validation_end,test_start
0,final_holdout_after_all_validation_periods,PASS,2024-12-31 23:00:00,2025-01-01


In [8]:
# Validate the preprocessing policy using the configuration
preprocessing_validation = validate_preprocessing_policy(CONFIG)
preprocessing_validation


,check,status,configured_value
0,preprocessing_fit_scope_training_only,PASS,training_only


In [9]:
# Assert that all validations passed
assert (temporal_validation["status"] == "PASS").all()
assert (holdout_validation["status"] == "PASS").all()
assert (preprocessing_validation["status"] == "PASS").all()

print("All common Modeling Foundation policy checks passed.")


All common Modeling Foundation policy checks passed.
